In [66]:
# https://judge.nitro-ai.org/competitions/nitro/pre-iaio-2026/2/view

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import re
import sys

In [67]:
def parse_target(target_str):
    match = re.search(r'\((-?\d+),\s*(-?\d+)\)', target_str)
    if match:
        return (int(match.group(1)), int(match.group(2)))
    return None

def parse_vectors(vectors_str):
    matches = re.findall(r'\((-?\d+),\s*(-?\d+)\)', vectors_str)
    return [(int(x), int(y)) for x, y in matches]

sys.setrecursionlimit(50000)
class Game:
    def __init__(self, row):
        self.n = int(row['N'])
        self.b = int(row['B'])
        self.target = parse_target(row['Target'])
        self.vectors = parse_vectors(row['Vectors'])

    def _distance(self, p1, p2):
        return abs(p1[0]-p2[0]) + abs(p1[1]-p2[1])

    def _sum(self, p1, p2):
        return (p1[0]+p2[0], p1[1]+p2[1])
    
    def solve(self):
        cur, turn = (0, 0), 0
        points = [0, 0]
        while self.n > 0:
            best_i, best_score = -1, None
            for i, vec in enumerate(self.vectors):
                willbe = self._sum(cur, vec)
                if abs(willbe[0]) > self.b or abs(willbe[1]) > self.b:
                    continue
                score = self._distance(self.target, cur) - self._distance(self.target, willbe)
                if self.target == willbe:
                    return f"PLAYER {turn}"
                for j, op_vec in enumerate(self.vectors):
                    if j == i:
                        continue
                    op_willbe = self._sum(willbe, op_vec)
                    if self.target == op_willbe:
                        score = -float('inf')
                        break
                if best_i == -1 or best_score < score:
                    best_i, best_score = i, score
                    
            if best_i == -1:
                return f"PLAYER {1-turn}"

            vec = self.vectors.pop(best_i)
            self.n -= 1
            cur = self._sum(cur, vec)
            points[turn] += best_score
            turn = 1 - turn

        if points[0] > points[1]:
            return "PLAYER 0"
        elif points[0] < points[1]:
            return "PLAYER 1"
            
        return "TIE"

In [68]:
test_df = pd.read_csv('/kaggle/input/datasets/abukanabek/preiaio-2026-2/test_data.csv')

solutions = []
for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    game = Game(row)
    solutions.append(game.solve())

  0%|          | 0/50 [00:00<?, ?it/s]

In [69]:
submission = pd.read_csv('/kaggle/input/datasets/abukanabek/preiaio-2026-2/sample_output.csv')

submission['answer'] = solutions

submission.to_csv('submission.csv', index=False)
submission['answer'].value_counts()

answer
PLAYER 0    34
PLAYER 1    10
TIE          6
Name: count, dtype: int64